# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [1]:
from dataclasses import dataclass
import csv
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

..\..\..\..\..\..\assets\03-wine-quality\inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [2]:
with DATA_PATH.open(newline="", encoding="utf-8") as source:
    reader = csv.DictReader(source)
    rows = []
    for item in reader:
        rows.append(item)

row = rows[0]
print("Numero de filas:", len(rows))
print("Columnas:", list(row.keys()))


Numero de filas: 5
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?
sample_id
2. ¿Cuántas características numéricas hay?
11
3. ¿Qué columna extra añadirías para probar el contrato?
cualquier columna extra que añada deberia invalidar el contrato
4. ¿Qué valor de `ph` sería claramente inválido?
Uno fuera del rango de ph que existe, por ejemplo 30, o un número negativo

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [ ]:
class WineQualityRequestPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid")

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)

model_row = {}
for key in row:
    if key != "sample_id":
        model_row[key] = row[key]
request = WineQualityRequestPydantic.model_validate(model_row)


### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [4]:
try:
    WineQualityRequestPydantic.model_validate(model_row)
    print("Fila valida -> aceptada")
except ValidationError as error:
    print("Fila valida -> rechazada:", error)

row_with_extra = dict(model_row)
row_with_extra["unexpected_field"] = 1
try:
    WineQualityRequestPydantic.model_validate(row_with_extra)
    print("Columna extra -> aceptada")
except ValidationError as error:
    print("Columna extra -> rechazada:", error)

row_with_bad_ph = dict(model_row)
row_with_bad_ph["ph"] = 99.0
try:
    WineQualityRequestPydantic.model_validate(row_with_bad_ph)
    print("ph fuera de rango -> aceptada")
except ValidationError as error:
    print("ph fuera de rango -> rechazada:", error)


Fila valida -> aceptada
Columna extra -> rechazada: 1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value=1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
ph fuera de rango -> rechazada: 1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=99.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [ ]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float

dataclass_request = WineQualityRequestDataclass(
    fixed_acidity=7.4,
    volatile_acidity=0.7,
    citric_acid=0.0,
    residual_sugar=1.9,
    chlorides=0.076,
    free_sulfur_dioxide=11.0,
    total_sulfur_dioxide=34.0,
    density=0.9978,
    ph=99.0,
    sulphates=0.56,
    alcohol=9.4,
)
print("dataclass acepta ph=99.0:", dataclass_request.ph)


dataclass acepta ph=99.0: 99.0


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en la entrada y salida de la inferencia porque valida los datos y rechaza valores o campos incorrectos. Usaría `dataclass` para estructuras internas porque es sencilla y ligera.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [6]:
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

vector = []
for name in FEATURE_NAMES:
    vector.append(getattr(request, name))
print("Vector:", vector)


Vector: [7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?
Si
- ¿El clasificador detectaría necesariamente el error?
No necesariamente. Puede aceptar los 11 valores y producir una predicción incorrecta.
- ¿Cómo evitarías que cada script definiera un orden distinto?
Usando una clase/función que devuelva siempre los datos en el orden correcto, para despues validar

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [7]:
class WineQualityPredictionPydantic(BaseModel):
    sample_id: str = Field(min_length=1)
    quality_band: Literal["needs_review", "acceptable", "excellent"]
    confidence: float = Field(ge=0, le=1)
    model_version: str = Field(min_length=1)
    preprocessing_version: str = Field(min_length=1)

prediction = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="acceptable",
    confidence=0.85,
    model_version="wine-quality-rf-v1",
    preprocessing_version="wine-red-features-v1",
)
print(prediction)


sample_id='red-001' quality_band='acceptable' confidence=0.85 model_version='wine-quality-rf-v1' preprocessing_version='wine-red-features-v1'


## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.